In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.tools import tool, tool_node
from typing import Dict, Any
from tavily import TavilyClient
import json


tavily_client = TavilyClient()

@tool
def search_web(query: str) -> Dict[str, Any]:
    """search the web for information."""
    return tavily_client.search(query)

@tool
def get_music() -> dict:
    """gets the music from JSON dictionary"""
    with open('resources/music.json', 'r') as file: 
        data = json.load(file)
    
    return data['music_database']

In [3]:
from dataclasses import dataclass

@dataclass
class userRole: 
    user_role: str = "external"

In [4]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """dynamically call tools based on the runtime context."""

    user_role = request.runtime.context.user_role

    if user_role == "internal":
        pass
    else:
        tools = [search_web]
        request = request.override(tools=tools)

    return handler(request)


In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="claude-haiku-4-5",
    tools=[search_web, get_music],
    middleware=[dynamic_tool_call],
    context_schema=userRole
)

In [ ]:
# internal user example
from langchain.messages import HumanMessage

response = agent.invoke(
    {
        "messages": [HumanMessage(content="What are the genres of music present?")]
    },
    context={"user_role": "internal"}
)

print(response['messages'][-1].content)

Based on the music data, the genres of music present are:

1. **R&B** - Features artists like The Weeknd, Miguel, Mariah Carey, TLC, Ne-Yo, Khalid, John Legend, Beyoncé, Daniel Caesar, and H.E.R.

2. **Pop** - Features artists like Taylor Swift, Billie Eilish, Mark Ronson, Katy Perry, Dua Lipa, Harry Styles, Justin Bieber, Lady Gaga, and Justin Timberlake.

3. **Rap** - Features artists like Kendrick Lamar, Travis Scott, Eminem, Drake, Jay-Z, Kanye West, Future, and Lil Nas X.

4. **Rock** - Features artists like Queen, Nirvana, Eagles, Guns N' Roses, AC/DC, Led Zeppelin, Oasis, The White Stripes, and The Killers.

5. **Electronic** - Features artists like David Guetta, Martin Garrix, Major Lazer, Avicii, deadmau5, Swedish House Mafia, Zedd, Above & Beyond, and Skrillex.

Each genre contains 10 songs spanning different decades, from the 1970s to the 2020s.


In [7]:
# external user example
response = agent.invoke(
    {"messages": [HumanMessage(content="What are the genres of music present?")]},
    context={"user_role": "external"}
)

print(response['messages'][-1].content)

I'd be happy to help you with information about music genres, but I need a bit more context. Are you asking about:

1. **General music genres** - all the different types of music that exist?
2. **Genres in a specific playlist, album, or collection** - music from something particular you're referring to?
3. **Genres in a particular time period or culture**?
4. **Something else specific**?

Could you clarify what you're looking for?


In [8]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What are the genres of music present in the JSON file")]},
    context={"user_role": "external"}
)

print(response['messages'][-1].content)

I don't see any JSON file attached to your message. To help you identify the genres of music in a JSON file, I would need you to either:

1. **Share the JSON file content** - You can paste the JSON code directly in your message
2. **Upload the file** - If your platform supports file uploads, you can attach the JSON file
3. **Provide a link** - If the file is hosted online, you can share the URL

Once you provide the JSON file, I'll be happy to help you identify all the music genres present in it.
